# Домашняя работа 11. Полная проблема собственных значений (Метод вращений Якоби)
**Колпакова Дарья Сергеевна, группа J3112, ИСУ 501390**

**Вариант 11**

**Задание:**
Реализовать решение полной проблемы собственных чисел методом плоских вращений Якоби.
Критерий остановки: $\max_{i \neq j} |B_k(i, j)| < 10^{-5}$ (максимальный по модулю внедиагональный элемент матрицы $B_k$ на $k$-й итерации меньше заданного порога).

Исходные данные:
- Матрица $A = C^T C$, где $C$ — первые 13 столбцов файла `housing.csv`.

### Теоретическая справка

Метод плоских вращений Якоби применяется для нахождения всех собственных значений и собственных векторов симметричной матрицы с действительными элементами.
Идея метода состоит в последовательном обнулении максимального по модулю внедиагонального элемента матрицы $A_k$ с помощью ортогональных преобразований вращения:
$$A_{k+1} = S_k^T A_k S_k$$
где $S_k$ — матрица вращения, отличающаяся от единичной только четырьмя элементами в строках и столбцах с индексами $p$ и $q$:
- $S_{pp} = \cos \varphi$
- $S_{qq} = \cos \varphi$
- $S_{pq} = -\sin \varphi$
- $S_{qp} = \sin \varphi$

Индексы $p$ и $q$ выбираются так, чтобы обнулить максимальный внедиагональный элемент $a_{pq}^{(k)}$.
Угол $\varphi$ вычисляется через параметр $t$:
$$t = \frac{2a_{pq}^{(k)}}{a_{pp}^{(k)} - a_{qq}^{(k)}}$$
$$\cos \varphi = \sqrt{\frac{1}{2}\left(1 + \frac{1}{\sqrt{1+t^2}}\right)}$$
$$\sin \varphi = \text{sign}(t) \sqrt{\frac{1}{2}\left(1 - \frac{1}{\sqrt{1+t^2}}\right)}$$

Приближение к матрице собственных векторов накапливается как:
$$U_{k+1} = U_k S_k, \quad U_0 = E$$
Процесс продолжается, пока матрица $A_k$ не станет достаточно близка к диагональной. Собственные значения окажутся на главной диагонали матрицы $A_k$, а столбцы матрицы $U_k$ будут соответствовать собственным векторам.


In [11]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=6, suppress=True)

VARIANT = 11
TOLERANCE = 1e-5
MAX_ITER = 100000
CSV_PATH = "housing.csv"

# подготовка данных
def read_housing_data(path):
    for header_mode in ["infer", None]:
        df = pd.read_csv(path, header=header_mode)
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.shape[1] == 14:
            return numeric_df.to_numpy(dtype=float)
    raise ValueError("Не удалось получить 14 числовых столбцов.")

data = read_housing_data(CSV_PATH)

# матрица C - первые 13 столбцов
C = data[:, :13]
# симметричная матрица A = C^T * C
A = C.T @ C

print(f"Размерность матрицы A: {A.shape}")

Размерность матрицы A: (13, 13)


In [12]:
# метод вращений Якоби
def jacobi_eigenvalue_method(A, tol=1e-5, max_iter=100000):
    n = A.shape[0]
    Ak = A.copy()
    Uk = np.eye(n)

    for k in range(1, max_iter + 1):
        # ищем максимальный по модулю внедиагональный элемент
        # для этого зануляем диагональ во временной копии
        off_diag = np.abs(Ak - np.diag(np.diag(Ak)))
        p, q = np.unravel_index(np.argmax(off_diag), Ak.shape)

        max_val = off_diag[p, q]

        # критерий остановки
        if max_val < tol:
            return np.diag(Ak), Uk, k

        # вычисляем параметры вращения
        if Ak[p, p] == Ak[q, q]:
            c = np.cos(np.pi / 4)
            s = np.sin(np.pi / 4)
        else:
            t = 2 * Ak[p, q] / (Ak[p, p] - Ak[q, q])
            c = np.sqrt(0.5 * (1 + 1 / np.sqrt(1 + t**2)))
            s = np.sign(t) * np.sqrt(0.5 * (1 - 1 / np.sqrt(1 + t**2)))

        # строим матрицу вращения Sk
        Sk = np.eye(n)
        Sk[p, p] = c
        Sk[q, q] = c
        Sk[p, q] = -s
        Sk[q, p] = s

        # обновляем матрицы
        Ak = Sk.T @ Ak @ Sk
        Uk = Uk @ Sk

    print("Метод Якоби: достигнуто максимальное число итераций.")
    return np.diag(Ak), Uk, max_iter

In [13]:
# вычисления
print("\nМетод вращений Якоби")
eigenvalues_jacobi, eigenvectors_jacobi, iters = jacobi_eigenvalue_method(A, tol=TOLERANCE, max_iter=MAX_ITER)

if iters == MAX_ITER:
    print(f"Остановка по лимиту итераций ({iters}).")
else:
    print(f"Метод сошелся за {iters} итераций.")
print("\nСобственные значения (Метод Якоби):")
# сортируем по убыванию для удобства сравнения
idx_sort = np.argsort(eigenvalues_jacobi)[::-1]
eigenvalues_jacobi_sorted = eigenvalues_jacobi[idx_sort]
print(eigenvalues_jacobi_sorted)

# проверка через встроенную функцию
print("\nТочное решение (np.linalg.eigh)")
eigenvalues_exact, eigenvectors_exact = np.linalg.eigh(A)
# eigh возвращает значения по возрастанию, перевернем для сравнения
eigenvalues_exact_sorted = eigenvalues_exact[::-1]

print("Собственные значения (np.linalg.eigh):")
print(eigenvalues_exact_sorted)

# вычислим максимальную погрешность по собственным значениям
error = np.max(np.abs(eigenvalues_jacobi_sorted - eigenvalues_exact_sorted))
print(f"\nМаксимальная абсолютная погрешность собственных значений: {error:.6e}")


Метод вращений Якоби
Метод Якоби: достигнуто максимальное число итераций.
Остановка по лимиту итераций (100000).

Собственные значения (Метод Якоби):
[1.583868e+08 1.187474e+07 4.170022e+05 1.616446e+05 2.526975e+04
 1.476296e+04 8.183960e+03 6.073267e+03 4.235775e+03 6.063995e+02
 3.274126e+02 3.041579e+01 2.193277e+00]

Точное решение (np.linalg.eigh)
Собственные значения (np.linalg.eigh):
[1.583868e+08 1.187474e+07 4.170022e+05 1.616446e+05 2.526975e+04
 1.476296e+04 8.183960e+03 6.073267e+03 4.235775e+03 6.063995e+02
 3.274126e+02 3.041578e+01 2.193270e+00]

Максимальная абсолютная погрешность собственных значений: 1.552171e-05


Метод плоских вращений Якоби успешно реализован и применен к симметричной матрице $A = C^T C$.
Алгоритм итеративно обнулял максимальные внедиагональные элементы, стремясь к заданному порогу $\varepsilon = 10^{-5}$.

Из-за того, что исходная матрица $A$ содержит элементы огромного масштаба (порядка $10^8$), достижение абсолютной погрешности $10^{-5}$ для всех внедиагональных элементов требует колоссального числа итераций (относительная точность должна составить $\sim 10^{-13}$). Поэтому алгоритм останавливается по заданному лимиту в $100 000$ итераций.

Несмотря на остановку по лимиту, полученные собственные значения практически полностью совпадают с результатами встроенной библиотечной функции `np.linalg.eigh` (максимальная абсолютная погрешность составила около $1.5 \times 10^{-5}$). Это подтверждает корректность работы реализованного метода для решения полной проблемы собственных значений.